Notes : teori dapat dibaca di https://panoramic-sandpaper-925.notion.site/Session-2-DNA-Composition-Analysis-2fb1630ea7ca807cb2acf8a9e32aac0c?pvs=143**

# Session 2 DNA Composition Analysis

In [28]:
!pip install biopython

In [29]:
import Bio

In [30]:
from Bio.SeqUtils import gc_fraction
from Bio.Seq import Seq

In [31]:
seq = Seq('GCTAGCATTAGCTAGCCCGGTTAATTA')

##GC AT Composition

#####GC AT Using Biopython

In [32]:
# GC Composition
gc_composition = gc_fraction(seq)
print(f"GC Composition: {gc_composition}")

# AT Composition
at_composition = 1 - gc_composition
print(f"AT Composition: {at_composition}")

GC Composition: 0.4444444444444444
AT Composition: 0.5555555555555556


In [33]:
#Di quiz biasanya minta komposisi GC AT itu diminta dalam bentuk presentase, jadi jangan lupa untuk di convert ke persen
print(f"GC = {gc_composition * 100}%")
print(f"AT = {at_composition * 100}%")

GC = 44.44444444444444%
AT = 55.55555555555556%


In [34]:
#Versi pendek
print(f"GC = {gc_fraction(seq) * 100} %")
print(f"AT = {100 - (gc_fraction(seq) * 100)} %")

GC = 44.44444444444444 %
AT = 55.55555555555556 %


#####GC AT Using List Comprehesion (loop)

In [35]:
#ini cara manual looping dari sequencenya
def GC(seq):
  list = [x for x in seq if x == 'G' or x == 'C']
  return len(list) / len(seq)

print(f"GC Composition: {GC(seq)*100}%")
print(f"AT Composition: {(1 - GC(seq))*100}%")

GC Composition: 44.44444444444444%
AT Composition: 55.55555555555556%


##Melting Point

Simplenya inituh nyari pada suhu berapa sih sequence dna itu berpisah, karena sebelum di transcribe dia perlu misah dulu kan (lebih lengkapnya baca di notion)

In [36]:
from Bio.SeqUtils import MeltingTemp

####Wallace Rule -> biasanya keluar di quiz

In [37]:
# Melting Temperature Wallace (built in)
# Notes -> biasanya di soal minta pake built in function aja
mt_wallace = MeltingTemp.Tm_Wallace(seq) # ini di pc binus (kmg) kayaknya gaada auto completenya
print(f"Melting Temperature Wallace: {mt_wallace}")

Melting Temperature Wallace: 78.0


In [47]:
# Code dibalik MeltingTemp.Tm_Wallace (gausah diajarin gapapa)
# -> dia cuma pake rumus klasik tanpa mempertimbangkan panjang sequence
def TM_Wallace(seq):
    count_G = seq.count("G")
    count_C = seq.count("C")
    count_A = seq.count("A")
    count_T = seq.count("T")

    return 2 * (count_A + count_T) + 4 * (count_G + count_C)

print(f"Melting Temperature Wallace: {TM_Wallace(seq)}")

Melting Temperature Wallace: 78


In [38]:
#Function untuk cari melting temperature wallace rule secara manual
def calculate_wallace(seq):
  # counting nucleotides
  count_G = seq.count('G')
  count_C = seq.count('C')
  count_A = seq.count('A')
  count_T = seq.count('T')

  # ini di cek dulu, karena untuk sequence yang panjang rumusnya nanti beda
  # (menggunakan pendekatan empiris berbasis GC content)

  # sequence pendek (dibawah dan sama dengan 14 basa) -> menggunakan rumus klasik
  # Tm=2(A+T)+4(G+C)
  #   - Setiap A atau T menyumbang 2°C
  #   - Setiap G atau C menyumbang 4°C
  if len(seq) <= 14:
        return 2 * (count_A + count_T) + 4 * (count_G + count_C)

  # sequence panjang (diatas 14 basa) -> menggunakan rumus empiris khusus DNA panjang
  return 64.9 + 41 * (count_G + count_C - 16.4) / len(seq)

print(f"Melting Temperature Wallace: {calculate_wallace(seq)}")

Melting Temperature Wallace: 58.21851851851853




###### Rumus empiris khusus DNA panjang (Wallace long-DNA formula)

Tm = 64.9 + 41 * (G + C - 16.4) / N

---

###### Penjelasan Setiap Komponen

- **64.9**  : Konstanta dasar hasil pengamatan eksperimen untuk DNA panjang.

- **41**  : Faktor pengali yang menunjukkan kuatnya pengaruh kandungan GC terhadap kenaikan Tm.

- **G + C**  : Jumlah basa Guanin dan Sitosin dalam sekuens DNA.

- **16.4**  : Nilai koreksi empiris agar hasil perhitungan mendekati kondisi biologis nyata.

- **N**  : Panjang total sekuens DNA (jumlah seluruh basa).

**Kenapa hasil yang dari Biopython sama manual bisa beda?**
Karena function dari Biopython itu ga mempertimbangkan panjang dari sequencenya (jadi semuanya pake rumus klasik aja) sedangkan function yang kita buat itu mempertimbangkan panjang DNA nya (Wallace long-DNA formula)

**Kalo ditanya bagusan mana?** jawabannya bagusan pake function yang ada wallace long-dna formula, karena sebenernya wallace rule klasik itu cuma valid untuk dna yang sequence nya pendek, dan kalo dibandingin juga dia yang nilainya paling deket sama function melting temp yang lain


####Nearest Neighbor

In [39]:
# Melting Temperature Nerest Neighbor
mt_nn = MeltingTemp.Tm_NN(seq)
print(f"Melting Temperature Nearest Neighbor: {mt_nn}")

Melting Temperature Nearest Neighbor: 56.86079745126773


####GC

In [40]:
# Melting Temperature GC
mt_gc = MeltingTemp.Tm_GC(seq)
print(f"Melting Temperature GC: {mt_gc}")

Melting Temperature GC: 55.902902071977906


## Molecullar Weight

massa total satu sequence DNA, berdasarkan nukleotida penyusunnya, jadi konsepnya tuh jumlah setiap jenis basanya dikali berat setiap basanya,

MW = countA * massA + countC * massC + countG * massG + countT * massT

note:
massa yang digunakan bukan massa murni basa bebas, melainkan massa residu nukleotida dalam rantai DNA, karena saat nukleotida saling terhubung melalui ikatan fosfodiester, terjadi pelepasan molekul H₂O, sehingga massa setiap nukleotida dalam DNA lebih kecil dibandingkan massa nukleotida bebas.

In [41]:
from Bio.SeqUtils import molecular_weight

In [42]:
print(f"Molecular weight = {molecular_weight(seq)}")

Molecular weight = 8354.3326
